In [ ]:
# Importing the embeddings codes
from models.embeddings import get_embeddings

# Topic Modeling
### using Bert Sentence Embedding, LDA, and Clustering

In [ ]:
#importing all dependencies.
import os
import pandas as pd
import numpy as np
import re

import gensim
from gensim import corpora, models
from gensim.models.coherencemodel import CoherenceModel

from datetime import datetime

%matplotlib inline
import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import word_tokenize
from nltk.stem.porter import *

from collections import Counter

import umap
from wordcloud import WordCloud

import torch.nn.functional as F
import torch
from torch import nn

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

#### Text Related Utility Functions

In [ ]:
from utils.utility_functions import *

In [ ]:
def preprocess_word(s):
    """
    Get word level preprocessed data from preprocessed sentences
    including: remove punctuation, select noun, stem, stop_words
    :param s: sentence to be processed
    :return: word level pre-processed list
    """
    w_list = word_tokenize(s)
    w_list = f_punct(w_list)
    w_list = f_noun(w_list)
    w_list = f_stem(w_list)
    w_list = f_stopw(w_list)

    return w_list


In [ ]:
def preprocess_sent(t):
    """
    Get sentence level preprocessed data from raw texts
    :param t: review to be processed
    :return: text after sentence level preprocessing
    """
    s = sentence_clean(t)
    return s


In [ ]:
def preprocess_documents(docs, samp_size=None):
    """
    Preprocess the data
    """

    print('Preprocessing raw texts ...')
    n_docs = len(docs)
    
    sentences = []  # sentence level preprocessed
    token_lists = []  # word level preprocessed
    idx_in = []  # index of sample selected
    
    samp = np.random.choice(n_docs, samp_size)
    for i, idx in enumerate(samp):
        sentence = preprocess_sent(docs[idx])
        token_list = preprocess_word(sentence)

        if token_list:
            idx_in.append(idx)
            sentences.append(sentence)
            token_lists.append(token_list)
        print('{} %'.format(str(np.round((i + 1) / len(samp) * 100, 2))), end='\r')
    print('Preprocessing raw texts. Done!')
    return sentences, token_lists, idx_in


### Reading the data and preprocessing it

In [ ]:
file = 'data/sample_text.txt'
documents = pd.read_csv(file)
texts = documents.paragraph

samp_size = len(texts)

sentences, token_lists, idx_in = preprocess_documents(texts, samp_size=samp_size)

### Preparing autoencoder class

In [ ]:
from models.autoencoder_model import model as ae_model

class autoencoder():
    def __init__(self, latent_dim=32, epochs=10, batch_size=2):
        self.history = []
        self.model = ae_model(latent_dim)
        self.num_epochs = epochs
        
        
    def fit(self, X):
        
        learning_rate = 1e-3
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(self.model.parameters(), lr=learning_rate, weight_decay=1e-5)
        
        train_loss = []
        
        
        for epoch in range(self.num_epochs):

            for data in X:
                d = torch.tensor(data)
                d = torch.autograd.Variable(d)
                d = d.unsqueeze(0)
                
                # forward pass
                output = self.model(d.float())
                loss = criterion(output, d.float())

                
                # backward pass
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
    
            # log
            loss = loss.item()            
            train_loss.append(loss)
            print('Epoch {} of {}, Train Loss: {:.3f}'.format(epoch+1, self.num_epochs, loss))



### Preparing the Topic Model Class

In [ ]:
class Topic_Model:
    def __init__(self, k=3, method='LDA_BERT'):
        """
        :param k: number of topics
        :param method: method chosen for the topic model
        """
        if method not in {'LDA', 'BERT', 'LDA_BERT'}:
            raise Exception('Invalid method!')
        
        self.k = k
        self.dictionary = None
        self.corpus = None
        self.cluster_model = None
        self.ldamodel = None
        self.vec = {}
        self.gamma = 15  # parameter for reletive importance of lda
        self.method = method
        self.AE = None
        self.id = method + '_' + datetime.now().strftime("%Y_%m_%d_%H_%M_%S")


    def vectorize(self, sentences, token_lists, method=None):
        """
        Get vector representations from selected methods
        """
        
        # turn tokenized documents into a id <-> term dictionary
        self.dictionary = corpora.Dictionary(token_lists)
        # convert tokenized documents into a document-term matrix
        self.corpus = [self.dictionary.doc2bow(text) for text in token_lists]

        if method == 'LDA':
            print('Getting vector representations for LDA ...')
            if not self.ldamodel:
                self.ldamodel = gensim.models.ldamodel.LdaModel(self.corpus, num_topics=self.k, id2word=self.dictionary,
                                                                passes=20)

            def get_vec_lda(model, corpus, k):
                """
                Get the LDA vector representation (probabilistic topic assignments for all documents)
                :return: vec_lda with dimension: (n_doc * n_topic)
                """
                n_doc = len(corpus)
                vec_lda = np.zeros((n_doc, k))
                for i in range(n_doc):
                    # get the distribution for the i-th document in corpus
                    for topic, prob in model.get_document_topics(corpus[i]):
                        vec_lda[i, topic] = prob

                return vec_lda

            vec = get_vec_lda(self.ldamodel, self.corpus, self.k)
            print('Getting vector representations for LDA. Done!')
            return vec

        elif method == 'BERT':

            print('Getting vector representations for BERT ...')

            file = "data/sample_text.txt"
            bert_sentence_embedding = get_embeddings(file,'bert')
            bert_sentence_embedding = np.array(bert_sentence_embedding)

            vec = bert_sentence_embedding[idx_in]
            print('Getting vector representations for BERT. Done!')
            return vec
             
        elif method == 'LDA_BERT':
            
            
            vec_lda = self.vectorize(sentences, token_lists, method='LDA')

            vec_bert = self.vectorize(sentences, token_lists, method='BERT')
            
            vec_ldabert = np.c_[vec_lda * self.gamma, vec_bert]
            
            self.vec['LDA_BERT_FULL'] = vec_ldabert
            
            
            if not self.AE:
                # Learn the latent representation of data
                self.AE = autoencoder()
                print('Fitting Autoencoder ...')
                self.AE.fit(vec_ldabert)
                print('Fitting Autoencoder Done!')
            
            vec = self.AE.model.predict(vec_ldabert)
            return vec

    def fit(self, sentences, token_lists, method=None, m_clustering=None):
        """
        Fit the topic model for selected method given the preprocessed data
        :docs: list of documents, each doc is preprocessed as tokens
        :return:
        """

        #use kmeans clustering to cluster the latent representations
        m_clustering = KMeans
        self.cluster_model = m_clustering(self.k)

        print('Clustering the latent representations ...')
        
        self.vec[method] = self.vectorize(sentences, token_lists, self.method)
        
        self.cluster_model.fit(self.vec[method])
        
        print('Clustering. Done!')

    #not sure about usage of this function yet
    def predict(self, sentences, token_lists, out_of_sample=None):
        """
        Predict topics for new_documents
        """
        # Default as False
        out_of_sample = out_of_sample is not None

        if out_of_sample:
            corpus = [self.dictionary.doc2bow(text) for text in token_lists]
            if self.method != 'LDA':
                vec = self.vectorize(sentences, token_lists)
            
        else:
            corpus = self.corpus
            vec = self.vec.get(self.method, None)

        if self.method == "LDA":
            lbs = np.array(list(map(lambda x: sorted(self.ldamodel.get_document_topics(x),
                                                     key=lambda x: x[1], reverse=True)[0][0],
                                    corpus)))
        else:
            lbs = self.cluster_model.predict(vec)
        return lbs


## The Magic Begins here

In [ ]:
# Define the topic model object
method = "LDA_BERT"
ntopic = 3

TopicModelObject = Topic_Model(k = ntopic, method = method)

In [ ]:
# Fit the topic model
TopicModelObject.fit(sentences, token_lists)

In [ ]:
def get_topic_words(token_lists, labels, k=None):
    """
    get top words within each topic from clustering results
    """
    if k is None:
        k = len(np.unique(labels))
    topics = ['' for _ in range(k)]
    for i, c in enumerate(token_lists):
        topics[labels[i]] += (' ' + ' '.join(c))
    word_counts = list(map(lambda x: Counter(x.split()).items(), topics))
    # get sorted word counts
    word_counts = list(map(lambda x: sorted(x, key=lambda x: x[1], reverse=True), word_counts))
    # get topics
    topics = list(map(lambda x: list(map(lambda x: x[0], x[:10])), word_counts))

    return topics

In [ ]:
def get_silhouette(model):
    """
    Get silhouette score from model
    :param model: Topic_Model object
    :return: silhouette score
    """
    lbs = model.cluster_model.labels_
    vec = model.vec[model.method]
    return silhouette_score(vec, lbs)


In [ ]:


def plot_proj(embedding, lbs):
    """
    Plot UMAP embeddings
    :param embedding: UMAP (or other) embeddings
    :param lbs: labels
    """
    n = len(embedding)
    counter = Counter(lbs)
    for i in range(len(np.unique(lbs))):
        plt.plot(embedding[:, 0][lbs == i], embedding[:, 1][lbs == i], '.', alpha=0.5,
                 label='cluster {}: {:.2f}%'.format(i, counter[i] / n * 100))
    plt.legend(loc = 'best')
    plt.grid(color ='grey', linestyle='-',linewidth = 0.25)





In [ ]:

def get_wordcloud(model, token_lists, topic):
    """
    Get word cloud of each topic from fitted model
    :param model: Topic_Model object
    :param sentences: preprocessed sentences from docs
    """
    if model.method == 'LDA':
        return
    print('Getting wordcloud for topic {} ...'.format(topic))
    lbs = model.cluster_model.labels_
    tokens = ' '.join([' '.join(_) for _ in np.array(token_lists)[lbs == topic]])

    wordcloud = WordCloud(width=800, height=560,
                          background_color='white', collocations=False,
                          min_font_size=10).generate(tokens)

    # plot the WordCloud image
    plt.figure(figsize=(8, 5.6), facecolor=None)
    plt.imshow(wordcloud)
    plt.axis("off")
    plt.tight_layout(pad=0)
    dr = 'data/LDA_BERT/{}/{}'.format(model.method, model.id)
    if not os.path.exists(dr):
        os.makedirs(dr)
    plt.savefig('data/LDA_BERT' + '/Topic' + str(topic) + '_wordcloud')
    print('Getting wordcloud for topic {}. Done!'.format(topic))

In [ ]:

def get_coherence(model, token_lists, measure='c_v'):
    """
    Get model coherence from gensim.models.coherencemodel
    :param model: Topic_Model object
    :param token_lists: token lists of docs
    :param topics: topics as top words
    :param measure: coherence metrics
    :return: coherence score
    """

    topics = get_topic_words(token_lists, model.cluster_model.labels_)
    cm = CoherenceModel(topics=topics, texts=token_lists, corpus=model.corpus, dictionary=model.dictionary,
                            coherence=measure)
    return cm.get_coherence()


In [ ]:
def visualize(model):
    """
    Visualize the result for the topic model by 2D embedding (UMAP)
    :param model: Topic_Model object
    """
    import umap
    if model.method == 'LDA':
        return
    reducer = umap.UMAP()
    print('Calculating UMAP projection ...')
    vec_umap = reducer.fit_transform(model.vec[model.method])
    print('Calculating UMAP projection. Done!')
    plot_proj(vec_umap, model.cluster_model.labels_)
    dr = 'data/{}/{}'.format(model.method, model.id)
    print (dr)
    if not os.path.exists(dr):
        os.makedirs(dr)
    plt.savefig('2D_vis')

In [ ]:

print('Coherence:', get_coherence(tm, token_lists, 'c_v'))
print('Silhouette Score:', get_silhouette(tm))
# visualize and save img
visualize(tm)
for i in range(tm.k):
    get_wordcloud(tm, token_lists, i)

